# NLP Text Classification (Spam Detection)

Classifying SMS messages into Spam or Ham using NLP techniques and machine learning models.

In [ ]:
from google.colab import files
import os

files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

print("Kaggle API ready")

Saving kaggle.json to kaggle.json
Kaggle API ready


In [ ]:
!kaggle datasets download -d uciml/sms-spam-collection-dataset
!unzip sms-spam-collection-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset
License(s): unknown
100% 211k/211k [00:00<00:00, 369kB/s]

Archive:  sms-spam-collection-dataset.zip
  inflating: spam.csv                


In [ ]:
import pandas as pd

df = pd.read_csv("spam.csv", encoding="latin-1")

df = df[['v1', 'v2']]
df.columns = ['label', 'text']

df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Text Preprocessing

We clean the text data by:
- Converting to lowercase
- Removing punctuation
- Removing stopwords
- Applying stemming

In [ ]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

nltk.download('stopwords')

ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub('[^a-z]', ' ', text)  # remove punctuation/numbers
    text = text.split()
    text = [ps.stem(word) for word in text if word not in stop_words]
    return " ".join(text)

df['clean_text'] = df['text'].apply(clean_text)

df.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,label,text,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkt st m...
3,ham,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


## Feature Extraction using TF-IDF

Converting text into numerical features using TF-IDF vectorization.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df['clean_text']).toarray()
y = df['label'].map({'ham': 0, 'spam': 1})

print("Feature shape:", X.shape)

Feature shape: (5572, 3000)


## Train-Test Split

Split the dataset into training and testing sets to evaluate model performance fairly.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (4457, 3000)
Testing shape: (1115, 3000)


## Logistic Regression Model

Training a Logistic Regression model for binary classification (spam detection).

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

## Model Evaluation

Model Evaluation using precision, recall, and F1-score.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_log))

Accuracy: 0.9560538116591928

Classification Report:

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       0.96      0.70      0.81       150

    accuracy                           0.96      1115
   macro avg       0.96      0.85      0.89      1115
weighted avg       0.96      0.96      0.95      1115



## Naive Bayes Model

Then Comparing Logistic Regression with Naive Bayes for performance evaluation.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.9757847533632287

Classification Report:

              precision    recall  f1-score   support

           0       0.97      1.00      0.99       965
           1       1.00      0.82      0.90       150

    accuracy                           0.98      1115
   macro avg       0.99      0.91      0.94      1115
weighted avg       0.98      0.98      0.97      1115



## Model Comparison

Comparing Logistic Regression and Naive Bayes based on performance metrics.

In [ ]:
print("LOGISTIC REGRESSION RESULTS")
print(classification_report(y_test, y_pred_log))

print("\nNAIVE BAYES RESULTS")
print(classification_report(y_test, y_pred_nb))

LOGISTIC REGRESSION RESULTS
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       0.96      0.70      0.81       150

    accuracy                           0.96      1115
   macro avg       0.96      0.85      0.89      1115
weighted avg       0.96      0.96      0.95      1115


NAIVE BAYES RESULTS
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       965
           1       1.00      0.82      0.90       150

    accuracy                           0.98      1115
   macro avg       0.99      0.91      0.94      1115
weighted avg       0.98      0.98      0.97      1115



## Conclusion

Built an NLP classification model for spam detection.

### Key Steps:
- Text preprocessing (stopwords removal, stemming)
- TF-IDF feature extraction
- Logistic Regression and Naive Bayes models
- Model evaluation using precision, recall, and F1-score

### Final Insight:
Both models performed well, with Logistic Regression generally providing slightly better balance between precision and recall.